In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from surprise import Dataset, Reader, KNNBasic, SVD, accuracy
from surprise.model_selection import train_test_split as train_test_split_surprise
from surprise.model_selection import cross_validate
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as train_test_split_sklearn

In [86]:
point = (
    '../data/raw/movies.csv'
)
datos_movies = pd.read_csv(point, sep=',')
datos_movies.head(3)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [87]:
point = (
    '../data/raw/ratings.csv'
)
datos_ratings = pd.read_csv(point, sep=',')
datos_ratings.head(3)

,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246


In [88]:
datos_movies["genres"] = datos_movies["genres"].str.split("|")

df_exploded = datos_movies.explode('genres')
genre_dummies = pd.get_dummies(df_exploded['genres'])

df_combined = pd.concat([df_exploded[['movieId', 'title']], genre_dummies], axis=1)

In [89]:
df_final_movies = df_combined.groupby(['movieId', 'title'], as_index=False).sum()

In [90]:
df_final_movies

,movieId,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10324,146684,Cosmic Scrat-tastrophe (2015),0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10325,146878,Le Grand Restaurant (1966),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10326,148238,A Very Murray Christmas (2015),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
10327,148626,The Big Short (2015),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [91]:
df = merged_data = pd.merge(
    datos_ratings,
    df_final_movies,
    on="movieId",
    how="right"
)
df.head()

,userId,movieId,rating,timestamp,title,(no genres listed),Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,2.0,1,5.0,8.590469e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,5.0,1,4.0,1.303501e+09,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,8.0,1,5.0,8.586109e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,11.0,1,4.0,8.508158e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,14.0,1,4.0,8.517663e+08,Toy Story (1995),0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [92]:
df['year'] = df['title'].str.rsplit('(', n=1).str[-1].str.replace(')', '')
df['title'] = df['title'].str.rsplit('(', n=1).str[0].str.strip()

df['year'] = df['year'].where(df['year'].str.isnumeric(), None).astype('Int64')

print(df)

        userId  movieId  rating     timestamp                         title  \
0          2.0        1     5.0  8.590469e+08                     Toy Story   
1          5.0        1     4.0  1.303501e+09                     Toy Story   
2          8.0        1     5.0  8.586109e+08                     Toy Story   
3         11.0        1     4.0  8.508158e+08                     Toy Story   
4         14.0        1     4.0  8.517663e+08                     Toy Story   
...        ...      ...     ...           ...                           ...   
105338   475.0   148238     3.0  1.451213e+09       A Very Murray Christmas   
105339   458.0   148626     4.0  1.452015e+09                 The Big Short   
105340   576.0   148626     4.5  1.451688e+09                 The Big Short   
105341   668.0   148626     4.5  1.451148e+09                 The Big Short   
105342   475.0   149532     4.0  1.451223e+09  Marco Polo: One Hundred Eyes   

        (no genres listed)  Action  Adventure  Anim

In [93]:
df = df.dropna()  # axis=0 por defecto
print(df.isna().sum())

userId                0
movieId               0
rating                0
timestamp             0
title                 0
(no genres listed)    0
Action                0
Adventure             0
Animation             0
Children              0
Comedy                0
Crime                 0
Documentary           0
Drama                 0
Fantasy               0
Film-Noir             0
Horror                0
IMAX                  0
Musical               0
Mystery               0
Romance               0
Sci-Fi                0
Thriller              0
War                   0
Western               0
year                  0
dtype: int64


In [94]:
datos_ratings['userId'].nunique()

668

In [95]:
df_colab  = datos_ratings.groupby('movieId')['rating'].mean().reset_index()

# opcional: renombrar la columna para mayor claridad
mean_ratings = df_colab.rename(columns={'rating': 'rating_medio'})

mean_ratings.head(20)

,movieId,rating_medio
0,1,3.907328
1,2,3.353261
2,3,3.189655
3,4,2.818182
4,5,3.250000
5,6,4.073913
6,7,3.381818
7,8,3.666667
8,9,2.869565
9,10,3.600000


In [96]:
X = mean_ratings
y = mean_ratings

In [97]:
#df_finito = df[['userId', 'movieId', 'rating']]
reader = Reader(rating_scale=(1, 5))
df_finito = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

In [98]:
trainset, testset = train_test_split_surprise(df_finito, test_size=0.2, random_state=42)

In [99]:
df_finito

In [100]:
def objective(trial):
    n_factors = trial.suggest_int('n_factors', 20, 200)
    reg_all = trial.suggest_float('reg_all', 0.001, 0.1, log=True)
    lr_all = trial.suggest_float('lr_all', 0.001, 0.1, log=True)

    model = SVD(n_factors=n_factors, reg_all=reg_all, lr_all=lr_all)
    cross_val_result = cross_validate(model, df_finito, measures=['RMSE'], cv=5, verbose=False)

    return cross_val_result['test_rmse'].mean()

In [101]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
best_params = study.best_params

[I 2025-05-23 19:16:24,769] A new study created in memory with name: no-name-9d9a5c6c-f835-4e06-b10a-9322dde183d6
[I 2025-05-23 19:16:31,934] Trial 0 finished with value: 0.8817718156983118 and parameters: {'n_factors': 115, 'reg_all': 0.09920075541894038, 'lr_all': 0.001701317562372154}. Best is trial 0 with value: 0.8817718156983118.
[I 2025-05-23 19:16:38,598] Trial 1 finished with value: 0.8757676385367986 and parameters: {'n_factors': 143, 'reg_all': 0.015785391006111232, 'lr_all': 0.04819393191402378}. Best is trial 1 with value: 0.8757676385367986.
[I 2025-05-23 19:16:42,197] Trial 2 finished with value: 0.8864964822485888 and parameters: {'n_factors': 20, 'reg_all': 0.05469104535968797, 'lr_all': 0.039012824087632965}. Best is trial 1 with value: 0.8757676385367986.
[I 2025-05-23 19:16:47,965] Trial 3 finished with value: 0.8850117038948822 and parameters: {'n_factors': 116, 'reg_all': 0.018158695732527184, 'lr_all': 0.023398166189817268}. Best is trial 1 with value: 0.87576763

In [109]:
#Aun no he configurado para que traiga los mejores parametros
sim_options = {'name': 'cosine', 'user_based': True}  # Cambia user_based=True para KNN de usuarios
model = SVD(n_factors=best_params['n_factors'], reg_all=best_params['reg_all'], lr_all=best_params['lr_all'])  # Esta linea dedicada para ajustar el consumo de RAM
#model = KNNBasic(sim_options=sim_options) #Errores por tamaño de dataset
model.fit(trainset)

predictions = model.test(testset)#predecimos

In [103]:
rmse = accuracy.rmse(predictions)
print(f'RMSE: {rmse}')

RMSE: 0.8676
RMSE: 0.8675856227938576


In [104]:

user = str(96737)  # IDs deben ser strings en Surprise.... por alguna razón 
item = str(242)
pred = model.predict(user, item)
print(f'Predicción  {user} para peli {item}',pred)

Predicción  96737 para peli 242 user: 96737      item: 242        r_ui = None   est = 3.52   {'was_impossible': False}


In [105]:
predictions

[Prediction(uid=198.0, iid=2971, r_ui=3.0, est=4.050205531012718, details={'was_impossible': False}),
 Prediction(uid=192.0, iid=3148, r_ui=4.0, est=3.8223623225283543, details={'was_impossible': False}),
 Prediction(uid=96.0, iid=736, r_ui=3.0, est=2.6629921971989887, details={'was_impossible': False}),
 Prediction(uid=615.0, iid=1645, r_ui=4.0, est=3.8051821845596376, details={'was_impossible': False}),
 Prediction(uid=140.0, iid=1006, r_ui=2.0, est=2.934218813594623, details={'was_impossible': False}),
 Prediction(uid=164.0, iid=1485, r_ui=4.0, est=3.7694707646642467, details={'was_impossible': False}),
 Prediction(uid=387.0, iid=64969, r_ui=4.0, est=3.8821876392611316, details={'was_impossible': False}),
 Prediction(uid=668.0, iid=282, r_ui=3.5, est=2.839163551552413, details={'was_impossible': False}),
 Prediction(uid=434.0, iid=60069, r_ui=1.5, est=3.8293827788226946, details={'was_impossible': False}),
 Prediction(uid=567.0, iid=2021, r_ui=2.5, est=3.3684555235141524, details={'

In [115]:
df[df["movieId"] == 1006].head(1)

,userId,movieId,rating,timestamp,title,(no genres listed),Action,Adventure,Animation,Children,...,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
25059,140.0,1006,2.0,850121331.0,"Chamber, The",0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1996


In [117]:
pd.set_option('display.max_columns', None)

# Mostrar la fila 25059
df.iloc[25059:25060]  

,userId,movieId,rating,timestamp,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
25059,140.0,1006,2.0,850121331.0,"Chamber, The",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1996


In [122]:
df[(df["userId"] == 140.0) & (df["Drama"] == 1) & (df["movieId"] == 1006)]

,userId,movieId,rating,timestamp,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year
25059,140.0,1006,2.0,850121331.0,"Chamber, The",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1996
